In [1]:
import tokenizers
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torchmetrics
import transformers
from torch.utils.data import DataLoader
from datasets import load_dataset
import matplotlib.pyplot as plt
import gc
from collections import namedtuple

In [2]:
device = 'cuda'

In [3]:
plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

In [4]:
def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=2, factor=0.5, epoch_callback=None):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=patience, factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        if epoch_callback is not None:
            epoch_callback(model, epoch)
        for index, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
            train_metric = metric.compute().item()
            print(f"\rBatch {index + 1}/{len(train_loader)}", end="")
            print(f", loss={total_loss/(index+1):.4f}", end="")
            print(f", {train_metric=:.2%}", end="")
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(f"\rEpoch {epoch + 1}/{n_epochs},                      "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.2%}, "
              f"valid metric: {history['valid_metrics'][-1]:.2%}")
    return history


In [5]:

def del_vars(variable_names= []):
    for name in variable_names:
        try:
            del globals()[name]
        except KeyError:
            pass
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

In [6]:
nmt_original_valid_set, nmt_test_set = load_dataset(
    path="ageron/tatoeba_mt_train", name = "eng-spa",
    split = ["validation", "test"]
)
split = nmt_original_valid_set.train_test_split(train_size=0.8, seed=42)
print(split)
nmt_train_set, nmt_valid_set = split["train"], split["test"]

DatasetDict({
    train: Dataset({
        features: ['source_text', 'target_text', 'source_lang', 'target_lang'],
        num_rows: 157839
    })
    test: Dataset({
        features: ['source_text', 'target_text', 'source_lang', 'target_lang'],
        num_rows: 39460
    })
})


In [7]:
print(type(nmt_train_set))
print(nmt_train_set[0])

<class 'datasets.arrow_dataset.Dataset'>
{'source_text': 'Tom tried to break up the fight.', 'target_text': 'Tom trató de disolver la pelea.', 'source_lang': 'eng', 'target_lang': 'spa'}


In [8]:
def train_eng_spa():
    for pair in nmt_train_set:
        yield pair["source_text"]
        yield pair["target_text"]

max_length = 256
vocab_size = 10_000

nmt_tokenizer_model = tokenizers.models.BPE(unk_token="<unk>")
nmt_tokenizer = tokenizers.Tokenizer(nmt_tokenizer_model)
nmt_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
nmt_tokenizer.enable_truncation(max_length=max_length)
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
nmt_tokenizer_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=vocab_size, special_tokens=["<pad>", "<unk>", "<s>", "</s>"])
nmt_tokenizer.train_from_iterator(train_eng_spa(), nmt_tokenizer_trainer)



In [9]:
nmt_tokenizer.encode("I like footbal").ids

[43, 401, 2993, 1452]

In [10]:
nmt_tokenizer.encode("<s> Me gusta el futbol").ids

[2, 396, 582, 219, 376, 3075]

In [11]:
from collections import namedtuple

fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"]
class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device):
        return NmtPair(self.src_token_ids.to(device), self.src_mask.to(device),
                       self.tgt_token_ids.to(device), self.tgt_mask.to(device))


In [12]:
def nmt_collate_fn(batch):
    src_texts = [pair['source_text'] for pair in batch]
    tgt_texts = [f"<s> {pair['target_text']} </s>" for pair in batch]
    src_encodings = nmt_tokenizer.encode_batch(src_texts)
    tgt_encodings = nmt_tokenizer.encode_batch(tgt_texts)
    src_token_ids = torch.tensor([enc.ids for enc in src_encodings])
    tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings])
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings])
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings])
    inputs = NmtPair(src_token_ids, src_mask,
                     tgt_token_ids[:, :-1], tgt_mask[:, :-1])
    labels = tgt_token_ids[:, 1:]
    return inputs, labels

batch_size = 32
nmt_train_loader = DataLoader(nmt_train_set, batch_size=batch_size,
                              collate_fn=nmt_collate_fn, shuffle=True)
nmt_valid_loader = DataLoader(nmt_valid_set, batch_size=batch_size,
                              collate_fn=nmt_collate_fn)
nmt_test_loader = DataLoader(nmt_test_set, batch_size=batch_size,
                             collate_fn=nmt_collate_fn)


In [61]:
class NmtModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=521, pad_id=0, hidden_dim=512, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.encoder = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        self.decoder = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, pair):
        src_embeddings = self.embed(pair.src_token_ids)
        tgt_embeddings = self.embed(pair.tgt_token_ids)
        src_lengths = pair.src_mask.sum(dim=1)
        src_packed = pack_padded_sequence(
            src_embeddings, lengths=src_lengths.cpu(),
            batch_first=True, enforce_sorted=False
        )
        _, hidden_states = self.encoder(src_packed)
        outputs, _ = self.decoder(tgt_embeddings, hidden_states)
        return self.output(outputs).permute(0,2,1)

In [62]:
torch.manual_seed(42)
vocab_size = nmt_tokenizer.get_vocab_size()
nmt_model = NmtModel(vocab_size).to(device)

In [63]:
n_epochs = 10
xentropy = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.NAdam(nmt_model.parameters(), lr=0.001)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=vocab_size)
accuracy = accuracy.to(device)
history = train(nmt_model, optimizer, xentropy, accuracy, nmt_train_loader, nmt_valid_loader, n_epochs)

Epoch 1/10,                      train loss: 3.1365, train metric: 17.37%, valid metric: 20.38%
Epoch 2/10,                      train loss: 2.0387, train metric: 22.01%, valid metric: 21.32%
Epoch 3/10,                      train loss: 1.7224, train metric: 23.44%, valid metric: 21.56%
Epoch 4/10,                      train loss: 1.5587, train metric: 24.31%, valid metric: 21.52%
Epoch 5/10,                      train loss: 1.4670, train metric: 24.62%, valid metric: 21.49%
Epoch 6/10,                      train loss: 1.4138, train metric: 24.88%, valid metric: 21.37%
Epoch 7/10,                      train loss: 1.0855, train metric: 27.06%, valid metric: 22.24%
Epoch 8/10,                      train loss: 0.8574, train metric: 28.79%, valid metric: 22.23%
Epoch 9/10,                      train loss: 0.7401, train metric: 29.67%, valid metric: 22.20%
Epoch 10/10,                      train loss: 0.6565, train metric: 30.38%, valid metric: 22.17%


In [18]:
def translate(model, src_text, max_length=20, pad_id=0, eos_id=3):
    tgt_text = ""
    token_ids = []
    for index in range(max_length):
        batch, _ = nmt_collate_fn([{"source_text": src_text,
                                    "target_text": tgt_text}])
        with torch.no_grad():
            Y_logits = model(batch.to(device))
            Y_token_ids = Y_logits.argmax(dim=1)  # find the best token IDs
            next_token_id = Y_token_ids[0, index]  # take the last token ID

        next_token = nmt_tokenizer.id_to_token(next_token_id)
        tgt_text += " " + next_token
        if next_token_id == eos_id:
            break
    return tgt_text

In [65]:
nmt_model.eval()
translate(nmt_model, "I like soccer.")

' Me gusta el fútbol . </s>'

In [ ]:
longer_text = "I like to paly socces with my friends. "
translate(nmt_model, longer_text)

' Me gustan los cone jos con toma clo de mis amigos . </s>'

In [67]:
import os

os.mkdir("models")

In [70]:
torch.save(nmt_model.state_dict(), "models/NMT-ENGLISH-TO-SPANISH.pt")

In [71]:
def beam_search(model, src_text, beam_width=3, max_length=20,
                verbose=False, length_penalty=0.6):
    top_translations = [(torch.tensor(0.), "")]
    for index in range(max_length):
        if verbose:
            print(f"Top {beam_width} translations so far:")
            for log_proba, tgt_text in top_translations:
                print(f"    {log_proba.item():.3f} – {tgt_text}")

        candidates = []
        for log_proba, tgt_text in top_translations:
            if tgt_text.endswith(" </s>"):
                candidates.append((log_proba, tgt_text))
                continue  # don't add tokens after EOS token
            batch, _ = nmt_collate_fn([{"source_text": src_text,
                                        "target_text": tgt_text}])
            with torch.no_grad():
                Y_logits = model(batch.to(device))
                Y_log_proba = F.log_softmax(Y_logits, dim=1)
                Y_top_log_probas = torch.topk(Y_log_proba, k=beam_width, dim=1)

            for beam_index in range(beam_width):
                next_token_log_proba = Y_top_log_probas.values[0, beam_index, index]
                next_token_id = Y_top_log_probas.indices[0, beam_index, index]
                next_token = nmt_tokenizer.id_to_token(next_token_id)
                next_tgt_text = tgt_text + " " + next_token
                candidates.append((log_proba + next_token_log_proba, next_tgt_text))

        def length_penalized_score(candidate, alpha=length_penalty):
            log_proba, text = candidate
            length = len(text.split())
            penalty = ((5 + length) ** alpha) / (6 ** alpha)
            return log_proba / penalty

        top_translations = sorted(candidates,
                                  key=length_penalized_score,
                                  reverse=True)[:beam_width]

    return top_translations[0][1]

In [73]:
beam_search(nmt_model, longer_Text, beam_width=3)

' Me gustan las frutas con mis amigos . </s>'

In [74]:
longest_text = "I like to play soccer with my friends at the beach."
beam_search(nmt_model, longest_text, beam_width=3)

' Me gusta jugar a jugar a la playa con la playa . </s>'

In [75]:
del_vars(["nmt_model"])

In [13]:
def attention(query, key, value):
    scores = query @ key.transpose(1,2)
    weights = torch.softmax(scores, dim=-1)
    return weights @ value

In [ ]:
class NmtModelWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim=512, pad_id=0, hidden_dim=512,
                 n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.encoder = nn.GRU(
            embed_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        self.decoder = nn.GRU(
            embed_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        self.output = nn.Linear(2 * hidden_dim, vocab_size)

    def forward(self, pair):
        src_embeddings = self.embed(pair.src_token_ids)
        tgt_embeddings = self.embed(pair.tgt_token_ids)
        src_lengths = pair.src_mask.sum(dim=1)
        src_packed = pack_padded_sequence(
            src_embeddings, lengths=src_lengths.cpu(),
            batch_first=True, enforce_sorted=False)
        encoder_outputs_packed, hidden_states = self.encoder(src_packed)
        decoder_outputs, _ = self.decoder(tgt_embeddings, hidden_states)
        encoder_outputs, _ = pad_packed_sequence(encoder_outputs_packed,
                                                 batch_first=True)
        attn_output = attention(query=decoder_outputs,
                                key=encoder_outputs, value=encoder_outputs)
        combined_output = torch.cat((attn_output, decoder_outputs), dim=-1)
        return self.output(combined_output).permute(0, 2, 1)


In [15]:
torch.manual_seed(42)
nmt_attn_model = NmtModelWithAttention(vocab_size).to(device)

n_epochs = 10
xentropy = nn.CrossEntropyLoss(ignore_index=0)  # ignore <pad> tokens
optimizer = torch.optim.NAdam(nmt_attn_model.parameters(), lr=0.001)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=vocab_size)
accuracy = accuracy.to(device)

history = train(nmt_attn_model, optimizer, xentropy, accuracy,
                nmt_train_loader, nmt_valid_loader, n_epochs)


Epoch 1/10,                      train loss: 3.0136, train metric: 18.02%, valid metric: 20.37%
Epoch 2/10,                      train loss: 2.1331, train metric: 21.53%, valid metric: 21.08%
Epoch 3/10,                      train loss: 1.9284, train metric: 22.47%, valid metric: 21.33%
Epoch 4/10,                      train loss: 1.8392, train metric: 22.85%, valid metric: 21.31%
Epoch 5/10,                      train loss: 1.7881, train metric: 23.15%, valid metric: 21.27%
Epoch 6/10,                      train loss: 1.7570, train metric: 23.27%, valid metric: 21.32%
Epoch 7/10,                      train loss: 1.4126, train metric: 25.12%, valid metric: 22.46%
Epoch 8/10,                      train loss: 1.1954, train metric: 26.47%, valid metric: 22.64%
Epoch 9/10,                      train loss: 1.0959, train metric: 27.04%, valid metric: 22.77%
Epoch 10/10,                      train loss: 1.0254, train metric: 27.52%, valid metric: 22.77%


In [16]:
torch.save(nmt_attn_model.state_dict(), "my_nmt_attn_model.pt")

In [19]:
longer_text = "I like to paly socces with my friends. "
translate(nmt_attn_model, longer_text)

' Me gusta usar car ros con mis amigos . </s>'